# 1. What is Data Leakage?

Data Leakage happens when information that should not be available during prediction is accidentally used by a machine learning model.

It can make a model show unrealistically high performance during training or testing, but perform poorly on new real-world data.

## AI/ML Example

Suppose an AI/ML Engineer builds a model to predict whether a customer will cancel a subscription.

Using a feature such as `Cancellation_Date` would cause data leakage because the cancellation information is only known after the customer cancels.

## Business Example

A bank wants to predict whether a customer will default on a loan.

Using `Default_Status` as an input feature would leak the answer into the model.

## Key Revision Point

Data Leakage means using information during model training that would not be available when making a real-world prediction.

In [2]:

## Simple Python Example

import pandas as pd

df = pd.DataFrame({
    "Age": [25, 30, 35],
        "Salary": [25000, 30000, 35000],
        "Purchased": [0, 1, 1]
})

X = df[["Age", "Salary", "Purchased"]]

print(X.columns.tolist())

['Age', 'Salary', 'Purchased']


# 2. Target Leakage

Target Leakage happens when information directly related to the target variable is included in the input features.

This gives the model information about the answer it is supposed to predict and can produce misleadingly high performance.


## What the Python Example Tells

The `Purchased` column is the target, but it has also been included in `X`.

This is target leakage because the model receives the answer as one of its input features.

## AI/ML Example

An AI/ML Engineer builds a customer churn prediction model.

Using `Cancellation_Date` as a feature would cause target leakage because the cancellation date is only known after the customer has already churned.

## Business Example

A bank predicts whether a customer will default on a loan.

Using `Default_Status` as an input feature would leak the actual answer into the model.

In [3]:
## Python Example

import pandas as pd

df = pd.DataFrame({
    "Age": [25, 30, 35, 40],
    "Salary": [25000, 30000, 35000, 40000],
    "Purchased": [0, 0, 1, 1]
})

X = df[["Age", "Salary", "Purchased"]]
y = df["Purchased"]

print("Features:", X.columns.tolist())
print("Target:", y.name)


Features: ['Age', 'Salary', 'Purchased']
Target: Purchased


# 3. Train-Test Contamination

Train-Test Contamination happens when information from the test dataset is used while preparing or training the model.

The test dataset should remain completely unseen until the final model evaluation.

## AI/ML Example

An AI/ML Engineer is building a customer prediction model.

If the engineer calculates preprocessing values using the complete dataset before splitting it into training and testing data, information from the test data can enter the training process.

## Business Example

A company builds a model to predict whether customers will buy a product.

The test customers must remain unseen while the model and preprocessing steps are prepared. Otherwise, the evaluation may show better performance than the model will achieve with new customers.

In [5]:

## Python Example

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.DataFrame({
    "Age": [20, 25, 30, 35, 40, 45],
    "Salary": [20000, 25000, 30000, 35000, 40000, 45000],
    "Purchased": [0, 0, 0, 1, 1, 1]
})

X = df[["Age", "Salary"]]
y = df["Purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 4
Testing rows: 2



## What the Python Example Tells

The data is first divided into training and testing sets.

The scaler learns the mean and standard deviation only from `X_train` using `fit_transform()`.

The same learned values are then used to transform `X_test` using `transform()`.

This prevents information from the test data from entering the training process.


# 4. Feature Leakage

Feature Leakage happens when a feature contains information that is only available after the event being predicted.

The feature may look useful to the model, but it would not be available when making a real-world prediction.

## AI/ML Example

An AI/ML Engineer builds a customer churn prediction model.

Using `Cancellation_Date` as an input feature would leak future information into the model.

## Business Example

A telecom company wants to predict which customers may leave.

Using the customer's account closure date would make the model look accurate, but that information is available only after the customer has already left.

In [6]:
## Python Example

import pandas as pd

df = pd.DataFrame({
    "Customer_Age": [25, 32, 40],
    "Monthly_Usage": [10, 20, 5],
    "Cancellation_Date": ["2026-01-10", "2026-02-15", "2026-03-20"],
    "Churn": [1, 1, 0]
})

X = df[["Customer_Age", "Monthly_Usage", "Cancellation_Date"]]

print(X.columns.tolist())


['Customer_Age', 'Monthly_Usage', 'Cancellation_Date']



## What the Python Example Tells

`Cancellation_Date` is included as a feature, but it is known only after a customer cancels.

Using this feature to predict `Churn` causes feature leakage because the model is receiving future information.


# 5. Preprocessing Leakage

Preprocessing Leakage happens when preprocessing steps use information from the test data before the model is evaluated.

The correct approach is to split the data first, then fit preprocessing steps only on the training data.

## AI/ML Example

An AI/ML Engineer standardizes features before training a model.

If `StandardScaler` is fitted on the complete dataset before splitting, information from the test data can enter the preprocessing process.

## Business Example

A bank builds a credit-risk model.

If missing values, scaling, or other preprocessing steps are calculated using both training and test customers, the test data is no longer completely unseen.

In [7]:

## Python Example

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.DataFrame({
    "Age": [20, 25, 30, 35, 40, 45],
    "Salary": [20000, 25000, 30000, 35000, 40000, 45000]
})

X_train, X_test = train_test_split(
    df, test_size=0.33, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training mean:", round(X_train["Age"].mean(), 2))
print("Test mean:", round(X_test["Age"].mean(), 2))


Training mean: 37.5
Test mean: 22.5


## What the Python Example Tells

The scaler learns preprocessing values only from the training data using `fit_transform()`.

The test data is transformed using those same learned values with `transform()`.

This prevents test-data information from influencing the preprocessing step.

## Output

Training mean: 30.0
Test mean: 40.0


# 6. Temporal Leakage

Temporal Leakage happens when a model uses information from the future to make a prediction about the past or present.

In time-based problems, the model should only use information that was available at the prediction time.

## AI/ML Example

An AI/ML Engineer builds a model to predict tomorrow's demand.

Using actual future sales, future customer activity, or future transaction information as input features would cause temporal leakage.

## Business Example

A retail company wants to predict today's product demand.

Using tomorrow's actual sales information would make the model appear highly accurate, but that information would not be available when today's prediction is made.

In [8]:

## Python Example

import pandas as pd

df = pd.DataFrame({
    "Date": ["2026-01-01", "2026-01-02", "2026-01-03"],
    "Sales": [100, 120, 150],
    "Next_Day_Sales": [120, 150, 180]
})

df["Date"] = pd.to_datetime(df["Date"])

X = df[["Sales", "Next_Day_Sales"]]

print(X.columns.tolist())

['Sales', 'Next_Day_Sales']



## What the Python Example Tells

`Next_Day_Sales` contains information from the future.

Using it to predict something at the current time causes temporal leakage because the next day's sales would not be known when making the current prediction.


# 7. Examples of Data Leakage

Data Leakage can occur in different ways when information that should not be available to the model enters the training data.

Common examples include using the target as a feature, using future information, and fitting preprocessing steps on the complete dataset.
## AI/ML Example

An AI/ML Engineer builds a customer purchase prediction model.

Using the customer's actual purchase status or information recorded after the purchase can cause data leakage.

## Business Example

An e-commerce company wants to predict whether a customer will purchase a product.

Using the customer's completed purchase information as an input feature would give the model information about the answer before prediction.

In [9]:

## Python Example

import pandas as pd

df = pd.DataFrame({
    "Age": [25, 30, 35],
    "Salary": [25000, 30000, 35000],
    "Purchased": [0, 1, 1],
    "Purchase_Date": ["2026-01-10", "2026-01-15", "2026-01-20"]
})

X = df[["Age", "Salary", "Purchased", "Purchase_Date"]]

print(X.columns.tolist())



['Age', 'Salary', 'Purchased', 'Purchase_Date']


# 8. How to Detect Leakage

Data Leakage can be detected by checking whether any feature contains information about the target that would not be available at prediction time.

A useful first step is to check the relationship between features and the target. An unusually strong relationship can be a warning sign of possible leakage.

## AI/ML Example

An AI/ML Engineer notices that one feature has an extremely strong relationship with the target and investigates whether the feature was created using future or target information.

## Business Example

A bank builds a loan-default prediction model.

If one feature almost perfectly predicts default, the data team should check whether that feature contains information recorded after the customer defaulted.

In [10]:

## Python Example

import pandas as pd

df = pd.DataFrame({
    "Age": [20, 25, 30, 35, 40],
    "Salary": [20000, 25000, 30000, 35000, 40000],
    "Purchased": [0, 0, 1, 1, 1]
})

correlation = df.corr(numeric_only=True)

print(correlation["Purchased"].round(2))

Age          0.87
Salary       0.87
Purchased    1.00
Name: Purchased, dtype: float64


# 9. How to Prevent Leakage

Data Leakage can be prevented by ensuring that the model only receives information that would be available at prediction time.

The main rule is to split the data before preprocessing and fit preprocessing steps only on the training data.

## AI/ML Example

An AI/ML Engineer should remove target-based and future-based features and perform preprocessing only after splitting the data.

## Business Example

A company building a customer churn model should use only information available before the prediction is made and keep future customer activity out of the model.


In [11]:

## Python Example

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.DataFrame({
    "Age": [20, 25, 30, 35, 40, 45],
    "Salary": [20000, 25000, 30000, 35000, 40000, 45000],
    "Purchased": [0, 0, 0, 1, 1, 1]
})

X = df[["Age", "Salary"]]
y = df["Purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))




Training rows: 4
Testing rows: 2



## What the Python Example Tells

The data is split before scaling.

`fit_transform()` is used only on the training data, so the scaler learns only from training information.

`transform()` is used on the test data without learning from it.

This keeps the test data completely unseen during preprocessing.

# 10. Incorrect Workflow and Correct Workflow

The workflow used to prepare data can cause data leakage if information from the test data is used before model evaluation.

The test data should remain unseen until the final evaluation.

## AI/ML Example

An AI/ML Engineer should split the dataset before scaling, encoding, feature selection, or other preprocessing steps.

## Business Example

A company building a customer churn model should train and preprocess using only historical training customers. Future test customers must remain unseen until evaluation.

In [12]:

## Python Example

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

df = pd.DataFrame({
    "Age": [20, 25, 30, 35, 40, 45],
    "Salary": [20000, 25000, 30000, 35000, 40000, 45000]
})

X_train, X_test = train_test_split(
    df, test_size=0.33, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaler fitted on training data:", True)
print("Test data transformed without fitting:", True)



Scaler fitted on training data: True
Test data transformed without fitting: True
